# 08 - Build and Verify Database

## Purpose
Define the relational schema, build the SQLite database, load all cleaned
datasets, and run a full verification suite to confirm row counts, join
integrity, foreign key compliance, and analytical gaps before the
calculation layer is built on top.

## Workflow
This notebook follows a strict schema-first, data-second pattern:
1. Load clean CSVs into DataFrames and prepare them for loading
2. Validate proposed primary keys against the actual data before committing
3. Define the full relational schema in SQL with explicit constraints
4. Build the database and load data into the declared schema
5. Run post-load integrity checks and analytical gap detection

## Inputs
- `data/clean/country_crosswalk.csv`
- `data/clean/cbam_defaults_clean.csv`
- `data/clean/eu_import_trade_flows_clean.csv`
- `data/clean/country_grid_electricity_clean.csv`
- `data/clean/hydrogen_route_intensities_clean.csv`
- `data/clean/steel_route_intensity_clean.csv`

## Output
- `db/cbam.db` — SQLite database with a fully declared relational schema

## Schema Overview

| Table | Role | Primary Key |
|---|---|---|
| `country_crosswalk` | Central country dimension | `country` |
| `cbam_defaults` | CBAM default emission values by country, CN code, and production route | `(country, cn_code, production_route_code)` |
| `trade_flows` | EU27 import flows by partner, CN code, year, and indicator | `(year, reporter, country, cn_code, indicator)` |
| `grid_co2_intensity` | Ember grid CO2 intensity by country and year | `(country, year)` |
| `grid_capacity` | Ember installed capacity by country, year, fuel type, subcategory, and unit | `(country, year, subcategory, fuel_type, unit)` |
| `grid_generation` | Ember electricity generation by country, year, fuel type, subcategory, and unit | `(country, year, subcategory, fuel_type, unit)` |
| `hydrogen_intensities` | JRC hydrogen route emission benchmarks | `(cn_code, feedstock_type)` |
| `steel_route_intensities` | Worldsteel production route emission benchmarks | `(production_route, year)` |

## Foreign Key Relationships
All country-level tables carry a foreign key on `country` referencing
`country_crosswalk(country)`. Foreign keys are declared in the schema for
diagram integrity and documentation, loaded with enforcement off for
performance, then verified post-load via explicit integrity queries.

## Notes
- `country` (canonical English name) is the join key across all country-level tables.
- `iso2` and `iso3` are retained in the crosswalk as secondary identifiers.
- The database is not committed to GitHub. It is regenerated from this
  notebook and the clean CSVs in `data/clean/`.
- `cbam_defaults` has legitimate duplicate `(country, cn_code)` combinations
  for cement CN codes 25231000 and 25239000, where grey and white clinker
  are assigned separate production routes per country. The third key column
  `production_route_code` disambiguates these rows. Non-cement rows have
  NULL `production_route_code` and are already unique on `(country, cn_code)`.
- `grid_capacity` and `grid_generation` require `unit` as a fifth key column
  because Ember reports generation in both TWh and % for the same
  `(country, year, subcategory, fuel_type)` combination.

---
## Section 1: Setup and Data Preparation

Load all clean CSVs, apply column renames for cross-table consistency,
and split the long-format Ember grid table into three purpose-built tables.
No database connection is opened in this section.

In [2]:
# ============================================================
# Imports and path configuration.
# sqlite3 is used for schema creation and PRAGMA control.
# pandas handles CSV loading and data preparation.
# sys is used to hard-stop execution if any pre-schema
# validation check fails, preventing a corrupt build.
# ============================================================
import sqlite3
import sys
import pandas as pd
from pathlib import Path

# Resolve paths relative to this notebook's location
clean  = Path("../data/clean")
db_path = Path("../db/cbam.db")
db_path.parent.mkdir(exist_ok=True)

print(f"Clean data directory : {clean.resolve()}")
print(f"Database output path : {db_path.resolve()}")

Clean data directory : /Users/milcahmaryjoseph/Documents/GitHub/cbam-analysis/data/clean
Database output path : /Users/milcahmaryjoseph/Documents/GitHub/cbam-analysis/db/cbam.db


In [3]:
# ============================================================
# Load all clean CSVs into DataFrames.
# Shape is printed for each to confirm row counts match
# the outputs of the upstream cleaning notebooks.
# ============================================================
crosswalk  = pd.read_csv(clean / 'country_crosswalk.csv', keep_default_na=False, na_values=[''])
defaults   = pd.read_csv(clean / "cbam_defaults_clean.csv")
flows      = pd.read_csv(clean / "eu_import_trade_flows_clean.csv")
grid       = pd.read_csv(clean / "country_grid_electricity_clean.csv")
hydrogen   = pd.read_csv(clean / "hydrogen_route_intensities_clean.csv")
steel      = pd.read_csv(clean / "steel_route_intensity_clean.csv")
global_exports   = pd.read_csv(clean / "comtrade_global_exports_clean.csv", dtype={"hs6_code": str, "iso3": str})
steel_route_mix  = pd.read_csv(clean / "steel_route_mix_clean.csv")

print("=== Raw Load Shapes ===")
print(f"  country_crosswalk : {crosswalk.shape}")
print(f"  cbam_defaults     : {defaults.shape}")
print(f"  trade_flows       : {flows.shape}")
print(f"  grid (raw)        : {grid.shape}")
print(f"  hydrogen          : {hydrogen.shape}")
print(f"  steel             : {steel.shape}")
print(f"  global_exports    : {global_exports.shape}")
print(f"  steel_route_mix   : {steel_route_mix.shape}")

=== Raw Load Shapes ===
  country_crosswalk : (240, 5)
  cbam_defaults     : (10671, 11)
  trade_flows       : (155848, 11)
  grid (raw)        : (193936, 10)
  hydrogen          : (6, 5)
  steel             : (12, 4)
  global_exports    : (14747, 10)
  steel_route_mix   : (119, 11)


In [7]:
# ============================================================
# Rename trade_flows columns for consistency with other tables.
#
#   product         -> cn_code         (matches cbam_defaults)
#   partner_country -> country         (matches crosswalk join key)
#   partner         -> iso2            (reflects actual content)
# ============================================================
flows = flows.rename(columns={
    "product":         "cn_code",
    "partner_country": "country",
    "partner":         "iso2",
})

print("trade_flows columns after rename:")
print(flows.columns.tolist())

trade_flows columns after rename:
['year', 'reporter', 'freq', 'iso2', 'country', 'flow', 'cn_code', 'indicator', 'indicator_label', 'value', 'material']


In [8]:
# ============================================================
# Split the Ember grid table into three purpose-built tables.
#
# The raw Ember CSV is long-format: Category and Variable columns
# combine CO2 intensity, capacity, and generation into one file.
# Splitting here creates three analytically distinct tables:
#
#   grid_co2_intensity  -- primary variable for CBAM indirect
#                          emissions calculations
#   grid_capacity       -- installed capacity by fuel type,
#                          retained for trend analysis
#   grid_generation     -- electricity generation by fuel type,
#                          retained for dashboard context
#
# Columns are renamed to lowercase snake_case to match the
# naming convention used across all other tables.
# ============================================================

# CO2 intensity: one row per country per year
grid_co2_intensity = grid[
    (grid["Category"] == "Power sector emissions") &
    (grid["Variable"] == "CO2 intensity")
][["country", "ISO 3 code", "Year", "Continent", "Ember region", "Value"]].copy()

grid_co2_intensity = grid_co2_intensity.rename(columns={
    "ISO 3 code":   "iso3",
    "Year":         "year",
    "Continent":    "continent",
    "Ember region": "ember_region",
    "Value":        "co2_intensity_gco2_kwh",
})

# Capacity: grain is (country, year, subcategory, fuel_type, unit)
grid_capacity = grid[
    grid["Category"] == "Capacity"
][["country", "ISO 3 code", "Year", "Subcategory", "Variable", "Unit", "Value"]].copy()

grid_capacity = grid_capacity.rename(columns={
    "ISO 3 code":  "iso3",
    "Year":        "year",
    "Subcategory": "subcategory",
    "Variable":    "fuel_type",
    "Unit":        "unit",
    "Value":       "value",
})

# Generation: grain is (country, year, subcategory, fuel_type, unit)
# Unit is required as a key column because Ember reports generation
# in both TWh and % for the same (country, year, subcategory, fuel_type)
grid_generation = grid[
    grid["Category"] == "Electricity generation"
][["country", "ISO 3 code", "Year", "Subcategory", "Variable", "Unit", "Value"]].copy()

grid_generation = grid_generation.rename(columns={
    "ISO 3 code":  "iso3",
    "Year":        "year",
    "Subcategory": "subcategory",
    "Variable":    "fuel_type",
    "Unit":        "unit",
    "Value":       "value",
})

print("=== Grid Split Shapes ===")
print(f"  grid_co2_intensity : {grid_co2_intensity.shape}")
print(f"  grid_capacity      : {grid_capacity.shape}")
print(f"  grid_generation    : {grid_generation.shape}")

=== Grid Split Shapes ===
  grid_co2_intensity : (5407, 6)
  grid_capacity      : (62141, 7)
  grid_generation    : (126388, 7)


---
## Section 2: Pre-Schema Validation

Before the database schema is written, every proposed primary key is
validated against the actual DataFrame data. This is the professional
pattern: assumptions are confirmed in code before they are committed
to a schema declaration.

If any check finds duplicate rows on a proposed key, execution halts
with a clear error message. A passing run means every `PRIMARY KEY`
declaration in Section 3 is backed by verified uniqueness.

In [5]:
# ============================================================
# Helper function: assert that a proposed composite key
# is unique across a DataFrame. Prints a pass/fail result
# and halts execution on failure.
# ============================================================
def assert_key_unique(df, key_cols, table_name):
    """
    Verify that the given columns form a unique key in the DataFrame.

    Parameters
    ----------
    df         : pd.DataFrame
    key_cols   : list of str  -- columns that form the proposed key
    table_name : str          -- used in pass/fail output only

    Behavior
    --------
    Prints a PASS line on success.
    Prints duplicate rows and calls sys.exit() on failure,
    stopping the notebook before any schema is written.
    """
    dupes = df[df.duplicated(subset=key_cols, keep=False)]
    if dupes.empty:
        key_str = " + ".join(key_cols)
        print(f"  PASS  {table_name:<30}  key: ({key_str})")
    else:
        print(f"\n  FAIL  {table_name}")
        print(f"  Proposed key columns: {key_cols}")
        print(f"  {len(dupes)} duplicate rows found:")
        print(dupes[key_cols].drop_duplicates().to_string())
        sys.exit(1)

In [9]:
# ============================================================
# Run uniqueness checks for all proposed primary keys.
#
# Each check corresponds directly to a PRIMARY KEY declaration
# in Section 3. All checks must pass before the schema is written.
#
# Note on cbam_defaults: the proposed key is
# (country, cn_code, production_route_code). Non-cement rows
# have NULL production_route_code. SQLite permits NULLs in
# composite primary keys, and pandas duplicated() treats NULLs
# as equal when checking for duplicates, so this check
# correctly catches any non-cement rows that share a
# (country, cn_code) pair.
# ============================================================
print("=== Pre-Schema Key Uniqueness Validation ===")
print()

assert_key_unique(
    crosswalk,
    ["country"],
    "country_crosswalk"
)

assert_key_unique(
    defaults,
    ["country", "cn_code", "production_route_code"],
    "cbam_defaults"
)

assert_key_unique(
    flows,
    ["year", "reporter", "country", "cn_code", "indicator"],
    "trade_flows"
)

assert_key_unique(
    grid_co2_intensity,
    ["country", "year"],
    "grid_co2_intensity"
)

assert_key_unique(
    grid_capacity,
    ["country", "year", "subcategory", "fuel_type", "unit"],
    "grid_capacity"
)

assert_key_unique(
    grid_generation,
    ["country", "year", "subcategory", "fuel_type", "unit"],
    "grid_generation"
)

assert_key_unique(
    hydrogen,
    ["cn_code", "feedstock_type"],
    "hydrogen_intensities"
)

assert_key_unique(
    steel,
    ["production_route", "year"],
    "steel_route_intensities"
)

assert_key_unique(
    global_exports,
    ["country", "hs6_code"],
    "global_exports"
)

assert_key_unique(
    steel_route_mix,
    ["country"],
    "steel_route_mix"
)

print()
print("All key uniqueness checks passed. Proceeding to schema definition.")

=== Pre-Schema Key Uniqueness Validation ===

  PASS  country_crosswalk               key: (country)
  PASS  cbam_defaults                   key: (country + cn_code + production_route_code)
  PASS  trade_flows                     key: (year + reporter + country + cn_code + indicator)
  PASS  grid_co2_intensity              key: (country + year)
  PASS  grid_capacity                   key: (country + year + subcategory + fuel_type + unit)
  PASS  grid_generation                 key: (country + year + subcategory + fuel_type + unit)
  PASS  hydrogen_intensities            key: (cn_code + feedstock_type)
  PASS  steel_route_intensities         key: (production_route + year)
  PASS  global_exports                  key: (country + hs6_code)
  PASS  steel_route_mix                 key: (country)

All key uniqueness checks passed. Proceeding to schema definition.


---
## Section 3: Schema Definition

All `CREATE TABLE` statements are declared here in a single cell.
This is the authoritative schema definition for the project database.

Design decisions:
- `country_crosswalk` is the central dimension table. All country-level
  fact tables carry a `FOREIGN KEY (country) REFERENCES country_crosswalk(country)`.
- `cbam_defaults`, `trade_flows`, `grid_capacity`, and `grid_generation`
  use composite primary keys reflecting the true analytical grain of each table.
- `hydrogen_intensities` and `steel_route_intensities` are benchmark/reference
  tables. They carry no foreign keys because their join logic to `cbam_defaults`
  is analytical rather than strict (route name mapping, CN code hierarchy).
- Foreign keys are declared for schema diagram integrity. Enforcement
  is handled post-load via explicit verification queries in Section 5.

In [10]:
# ============================================================
# Full schema definition as a list of CREATE TABLE statements.
#
# Tables are ordered so that referenced tables (dimensions)
# are created before tables that reference them (facts).
# This order matters when foreign_keys enforcement is ON.
# ============================================================
SCHEMA_SQL = [

    # ----------------------------------------------------------
    # DIMENSION: country_crosswalk
    # Central country identifier table. Every canonical country
    # name used across the project resolves through this table.
    # iso2 and iso3 are retained as secondary identifiers for
    # source-specific joins (COMEXT uses iso2, Ember uses iso3).
    # name_cbam_defaults and name_ember preserve source-specific
    # spellings for traceability.
    # ----------------------------------------------------------
    """
    CREATE TABLE IF NOT EXISTS country_crosswalk (
        country             TEXT NOT NULL,
        iso2                TEXT,
        iso3                TEXT,
        name_cbam_defaults  TEXT,
        name_ember          TEXT,
        PRIMARY KEY (country)
    );
    """,

    # ----------------------------------------------------------
    # FACT/REFERENCE: cbam_defaults
    # EU Commission default emission values by country, CN code,
    # and production route. The composite key includes
    # production_route_code to handle cement CN codes (25231000,
    # 25239000) that have two legitimate rows per country,
    # distinguishing grey clinker (A) from white clinker (B).
    # Non-cement rows have NULL production_route_code and are
    # already unique on (country, cn_code).
    # ----------------------------------------------------------
    """
    CREATE TABLE IF NOT EXISTS cbam_defaults (
        country                 TEXT NOT NULL,
        cn_code                 TEXT NOT NULL,
        description             TEXT,
        direct_emissions        REAL,
        indirect_emissions      REAL,
        total_emissions         REAL,
        default_2026            REAL,
        default_2027            REAL,
        default_2028_onwards    REAL,
        production_route_code   TEXT,
        production_route        TEXT,
        PRIMARY KEY (country, cn_code, production_route_code),
        FOREIGN KEY (country) REFERENCES country_crosswalk(country)
    );
    """,

    # ----------------------------------------------------------
    # FACT: trade_flows
    # EU27 import flows from COMEXT. One row per year, reporter,
    # partner country, CN code, and indicator. The indicator
    # column distinguishes value rows (VALUE_IN_EUROS) from
    # volume rows (QUANTITY_IN_KG) for the same trade pair.
    # ----------------------------------------------------------
    """
    CREATE TABLE IF NOT EXISTS trade_flows (
        year            INTEGER NOT NULL,
        reporter        TEXT    NOT NULL,
        freq            TEXT,
        iso2            TEXT,
        country         TEXT    NOT NULL,
        flow            TEXT,
        cn_code         TEXT    NOT NULL,
        indicator       TEXT    NOT NULL,
        indicator_label TEXT,
        value           REAL,
        material        TEXT,
        PRIMARY KEY (year, reporter, country, cn_code, indicator),
        FOREIGN KEY (country) REFERENCES country_crosswalk(country)
    );
    """,

    # ----------------------------------------------------------
    # FACT: grid_co2_intensity
    # Ember grid CO2 intensity in gCO2/kWh by country and year.
    # This is the primary variable used for CBAM indirect
    # emissions calculations. One row per country per year.
    # continent and ember_region are context columns retained
    # for geographic filtering in dashboards.
    # ----------------------------------------------------------
    """
    CREATE TABLE IF NOT EXISTS grid_co2_intensity (
        country                 TEXT    NOT NULL,
        iso3                    TEXT,
        year                    INTEGER NOT NULL,
        continent               TEXT,
        ember_region            TEXT,
        co2_intensity_gco2_kwh  REAL,
        PRIMARY KEY (country, year),
        FOREIGN KEY (country) REFERENCES country_crosswalk(country)
    );
    """,

    # ----------------------------------------------------------
    # FACT: grid_capacity
    # Ember installed electricity capacity in GW by country,
    # year, subcategory, fuel type, and unit. Unit is required
    # as a key column because Ember reports some capacity
    # figures in multiple units for the same combination.
    # ----------------------------------------------------------
    """
    CREATE TABLE IF NOT EXISTS grid_capacity (
        country     TEXT    NOT NULL,
        iso3        TEXT,
        year        INTEGER NOT NULL,
        subcategory TEXT    NOT NULL,
        fuel_type   TEXT    NOT NULL,
        unit        TEXT    NOT NULL,
        value       REAL,
        PRIMARY KEY (country, year, subcategory, fuel_type, unit),
        FOREIGN KEY (country) REFERENCES country_crosswalk(country)
    );
    """,

    # ----------------------------------------------------------
    # FACT: grid_generation
    # Ember electricity generation by country, year, subcategory,
    # fuel type, and unit. Unit is required as a key column
    # because Ember reports generation in both TWh and % for
    # the same (country, year, subcategory, fuel_type) combination.
    # ----------------------------------------------------------
    """
    CREATE TABLE IF NOT EXISTS grid_generation (
        country     TEXT    NOT NULL,
        iso3        TEXT,
        year        INTEGER NOT NULL,
        subcategory TEXT    NOT NULL,
        fuel_type   TEXT    NOT NULL,
        unit        TEXT    NOT NULL,
        value       REAL,
        PRIMARY KEY (country, year, subcategory, fuel_type, unit),
        FOREIGN KEY (country) REFERENCES country_crosswalk(country)
    );
    """,

    # ----------------------------------------------------------
    # REFERENCE: hydrogen_intensities
    # JRC hydrogen production route emission benchmarks.
    # Small reference table (6 rows). No foreign key is declared
    # because the relationship to cbam_defaults is analytical
    # (matched on cn_code ranges) rather than a strict FK join.
    # ----------------------------------------------------------
    """
    CREATE TABLE IF NOT EXISTS hydrogen_intensities (
        cn_code                     TEXT NOT NULL,
        feedstock_type              TEXT NOT NULL,
        total_emissions_tco2_per_th2 REAL,
        comments                    TEXT,
        source                      TEXT,
        PRIMARY KEY (cn_code, feedstock_type)
    );
    """,

    # ----------------------------------------------------------
    # REFERENCE: steel_route_intensities
    # Worldsteel production route emission benchmarks.
    # Small reference table (12 rows including Global avg).
    # No foreign key is declared because the join to
    # cbam_defaults uses a controlled route code mapping
    # rather than a direct key match.
    # ----------------------------------------------------------
    """
    CREATE TABLE IF NOT EXISTS steel_route_intensities (
        production_route        TEXT    NOT NULL,
        year                    INTEGER NOT NULL,
        co2_intensity_tco2_per_t REAL,
        energy_intensity_gj_per_t REAL,
        PRIMARY KEY (production_route, year)
    );
    """,

    # ----------------------------------------------------------
    # FACT: global_exports
    # UN Comtrade total export values and volumes by CBAM country
    # and 6-digit HS code. Used to compute CBAM cost as a share
    # of total export value -- the primary choropleth metric in
    # the project 01 dashboard.
    #
    # hs6_code is 6-digit HS (not 8-digit CN8). Multiple CN8
    # codes in cbam_defaults may map to the same hs6_code.
    # This is documented as a known limitation: some hs6_codes
    # may capture non-CBAM products sharing the same prefix.
    #
    # data_year records the actual year used where 2024 data was
    # unavailable and 2023 was substituted (is_fallback_year=1).
    # ----------------------------------------------------------
    """
    CREATE TABLE IF NOT EXISTS global_exports (
        country             TEXT    NOT NULL,
        iso2                TEXT,
        iso3                TEXT,
        hs6_code            TEXT    NOT NULL,
        sector              TEXT,
        export_tonnes       REAL,
        export_value_eur    REAL,
        year                INTEGER,
        data_year           INTEGER,
        is_fallback_year    INTEGER,
        PRIMARY KEY (country, hs6_code),
        FOREIGN KEY (country) REFERENCES country_crosswalk(country)
    );
    """,

    # ----------------------------------------------------------
    # REFERENCE: steel_route_mix
    # Worldsteel production route mix (BOF/EAF/OHF split) for
    # all 119 CBAM countries. Used to weight the BF-BOF and
    # Scrap-EAF intensity benchmarks when computing country-level
    # steel emission intensities.
    #
    # route_source distinguishes directly matched countries
    # ('direct') from those assigned a regional proxy
    # ('regional_proxy'). is_estimate=1 flags proxy rows.
    # production_mt is null for proxy rows (regional volume
    # is not meaningful at individual country level).
    # ----------------------------------------------------------
    """
    CREATE TABLE IF NOT EXISTS steel_route_mix (
        country             TEXT    NOT NULL,
        iso2                TEXT,
        iso3                TEXT,
        production_mt       REAL,
        bof_pct             REAL,
        eaf_pct             REAL,
        ohf_pct             REAL,
        bof_pct_combined    REAL,
        is_estimate         INTEGER,
        route_source        TEXT,
        proxy_region        TEXT,
        PRIMARY KEY (country),
        FOREIGN KEY (country) REFERENCES country_crosswalk(country)
    );
    """,

]

print(f"{len(SCHEMA_SQL)} CREATE TABLE statements defined.")
print("Schema is ready for execution.")

10 CREATE TABLE statements defined.
Schema is ready for execution.


---
## Section 4: Database Build and Data Load

The database is built in three steps:
1. Drop and recreate the database file cleanly to ensure a repeatable build
2. Execute all schema `CREATE TABLE` statements with foreign keys OFF
3. Load data using `to_sql` with `if_exists='append'`, which inserts into
   the already-declared schema rather than inferring column types

Foreign keys are kept OFF during the load for two reasons: load order
safety, and because the post-load verification queries in Section 5
provide a more informative integrity check than SQLite's constraint
errors would.

In [11]:
# ============================================================
# Drop the existing database if present, then create a fresh
# connection. This ensures the build is fully repeatable.
# Deleting the file rather than using DROP TABLE avoids any
# risk of stale schema fragments persisting between runs.
# ============================================================
if db_path.exists():
    db_path.unlink()
    print(f"Existing database removed: {db_path}")

conn = sqlite3.connect(db_path)
print(f"New database connection opened: {db_path}")

Existing database removed: ../db/cbam.db
New database connection opened: ../db/cbam.db


In [12]:
# ============================================================
# Execute the schema. Each CREATE TABLE statement is run
# individually so that any failure identifies the specific
# table rather than failing silently on a multi-statement
# executescript call.
#
# PRAGMA foreign_keys is set to OFF before schema execution
# and remains OFF through the data load in the next cell.
# It is turned ON in Section 5 for post-load verification.
# ============================================================
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = OFF;")

print("Executing schema...")
for statement in SCHEMA_SQL:
    cursor.execute(statement)

conn.commit()

# Confirm all 8 tables were created
tables_created = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    conn
)["name"].tolist()

print(f"\n{len(tables_created)} tables created:")
for t in tables_created:
    print(f"  {t}")

Executing schema...

10 tables created:
  cbam_defaults
  country_crosswalk
  global_exports
  grid_capacity
  grid_co2_intensity
  grid_generation
  hydrogen_intensities
  steel_route_intensities
  steel_route_mix
  trade_flows


In [13]:
# ============================================================
# Load all DataFrames into their corresponding tables.
#
# if_exists='append' is used instead of 'replace' because the
# schema already exists. Using 'replace' would drop and recreate
# the table without constraints, defeating the schema definition
# in Section 3.
#
# index=False prevents pandas from writing its integer index
# as an extra column.
# ============================================================

# Map DataFrame to target table name
load_plan = [
    (crosswalk,          "country_crosswalk"),
    (defaults,           "cbam_defaults"),
    (flows,              "trade_flows"),
    (grid_co2_intensity, "grid_co2_intensity"),
    (grid_capacity,      "grid_capacity"),
    (grid_generation,    "grid_generation"),
    (hydrogen,           "hydrogen_intensities"),
    (steel,              "steel_route_intensities"),
    (global_exports,  "global_exports"),
    (steel_route_mix, "steel_route_mix"),
]

print("Loading data...")
print()
for df, table_name in load_plan:
    df.to_sql(table_name, conn, if_exists="append", index=False)
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table_name};", conn).iloc[0]["n"]
    print(f"  {table_name:<30}  {count:>8,} rows loaded")

print()
print("All tables loaded.")

Loading data...

  country_crosswalk                    240 rows loaded
  cbam_defaults                     10,671 rows loaded
  trade_flows                      155,848 rows loaded
  grid_co2_intensity                 5,407 rows loaded
  grid_capacity                     62,141 rows loaded
  grid_generation                  126,388 rows loaded
  hydrogen_intensities                   6 rows loaded
  steel_route_intensities               12 rows loaded
  global_exports                    14,747 rows loaded
  steel_route_mix                      119 rows loaded

All tables loaded.


---
## Section 5: Post-Load Verification

Foreign keys are turned ON and a full verification suite is run:

1. **Foreign key integrity check** -- SQLite's built-in `PRAGMA foreign_key_check`
   reports any rows that violate a declared FK constraint
2. **Join integrity** -- left join checks confirm every country in
   `cbam_defaults` and `trade_flows` resolves to the crosswalk
3. **Trade flow to CBAM defaults match rate** -- confirms the expected
   31% match rate and documents why unmatched rows are expected
4. **Duplicate key investigation** -- confirms that the only duplicate
   `(country, cn_code)` pairs in `cbam_defaults` are the known cement routes
5. **Analytical gap detection** -- flags countries with no trade flow data
   or no grid CO2 intensity data, and confirms steel route coverage
6. **Summary** -- a single printed summary of all findings

In [14]:
# ============================================================
# Enable foreign key enforcement for post-load verification.
# PRAGMA foreign_key_check returns one row per violation.
# An empty result means all FK relationships are satisfied.
# ============================================================
conn.execute("PRAGMA foreign_keys = ON;")

print("Foreign key enforcement enabled.")
print()

# Run the built-in SQLite FK check across all tables
fk_violations = pd.read_sql("PRAGMA foreign_key_check;", conn)

if fk_violations.empty:
    print("PASS  PRAGMA foreign_key_check: no violations found across any table.")
else:
    print(f"FAIL  {len(fk_violations)} foreign key violation(s) found:")
    print(fk_violations.to_string(index=False))

Foreign key enforcement enabled.

PASS  PRAGMA foreign_key_check: no violations found across any table.


In [15]:
# ============================================================
# Join integrity check 1: cbam_defaults -> country_crosswalk
# Every country in cbam_defaults must resolve to a crosswalk row.
# Any unmatched country indicates a name that slipped through
# upstream cleaning.
# ============================================================
query = """
    SELECT d.country, COUNT(*) AS rows
    FROM cbam_defaults d
    LEFT JOIN country_crosswalk c ON d.country = c.country
    WHERE c.country IS NULL
    GROUP BY d.country
    ORDER BY d.country
"""
unmatched = pd.read_sql(query, conn)

if unmatched.empty:
    n_cbam = pd.read_sql(
        "SELECT COUNT(DISTINCT country) AS n FROM cbam_defaults;", conn
    ).iloc[0]["n"]
    print(f"PASS  cbam_defaults -> crosswalk: all {n_cbam} countries matched")
else:
    print(f"FAIL  cbam_defaults -> crosswalk: {len(unmatched)} unmatched country name(s):")
    print(unmatched.to_string(index=False))

PASS  cbam_defaults -> crosswalk: all 119 countries matched


In [16]:
# ============================================================
# Join integrity check 2: trade_flows -> country_crosswalk
# Every partner country in trade_flows must resolve to a
# crosswalk row. EU member states are present in trade_flows
# as reporters and partners, so this tests all 234 unique
# country values in that table.
# ============================================================
query = """
    SELECT t.country, COUNT(*) AS rows
    FROM trade_flows t
    LEFT JOIN country_crosswalk c ON t.country = c.country
    WHERE c.country IS NULL
    GROUP BY t.country
    ORDER BY t.country
"""
unmatched = pd.read_sql(query, conn)

if unmatched.empty:
    n_trade = pd.read_sql(
        "SELECT COUNT(DISTINCT country) AS n FROM trade_flows;", conn
    ).iloc[0]["n"]
    print(f"PASS  trade_flows -> crosswalk: all {n_trade} countries matched")
else:
    print(f"FAIL  trade_flows -> crosswalk: {len(unmatched)} unmatched country name(s):")
    print(unmatched.to_string(index=False))

PASS  trade_flows -> crosswalk: all 234 countries matched


In [17]:
# ============================================================
# Trade flows to CBAM defaults join rate.
#
# Not all trade flow rows are expected to match cbam_defaults.
# Unmatched rows fall into two legitimate categories:
#   1. EU member states: intra-EU trade is not subject to CBAM
#   2. Non-CBAM CN codes: most imported goods are outside CBAM scope
#
# Expected match rate is approximately 31%. A material deviation
# from this would indicate a join key or CN code format issue.
#
# Note: the join may produce more rows than trade_flows if a
# (country, cn_code) pair matches multiple cbam_defaults rows,
# as occurs for cement CN codes with two production routes.
# This is expected and documented in the notebook header.
# ============================================================
query = """
    SELECT
        COUNT(*)                                                    AS total_flow_rows,
        SUM(CASE WHEN d.country IS NOT NULL THEN 1 ELSE 0 END)      AS matched_rows,
        SUM(CASE WHEN d.country IS NULL     THEN 1 ELSE 0 END)      AS unmatched_rows
    FROM trade_flows t
    LEFT JOIN cbam_defaults d
        ON  t.country = d.country
        AND CAST(t.cn_code AS TEXT) = d.cn_code
"""
result = pd.read_sql(query, conn)

total    = result["total_flow_rows"].iloc[0]
matched  = result["matched_rows"].iloc[0]
pct      = matched / total * 100

print("Trade flows -> CBAM defaults join:")
print(result.to_string(index=False))
print(f"\nMatch rate: {pct:.1f}%  (expected ~31%)")

Trade flows -> CBAM defaults join:
 total_flow_rows  matched_rows  unmatched_rows
          156048         49490          106558

Match rate: 31.7%  (expected ~31%)


In [18]:
# ============================================================
# Confirm the only duplicate (country, cn_code) pairs in
# cbam_defaults are the known cement production route rows.
# Any new duplicate would indicate a data quality issue in
# an upstream cleaning step.
# ============================================================
query = """
    SELECT country, cn_code, COUNT(*) AS occurrences
    FROM cbam_defaults
    GROUP BY country, cn_code
    HAVING COUNT(*) > 1
    ORDER BY cn_code, country
"""
dupes = pd.read_sql(query, conn)

print(f"Duplicate (country, cn_code) pairs in cbam_defaults: {len(dupes)}")

if not dupes.empty:
    # All duplicates should be cement CN codes 25231000 and 25239000
    non_cement = dupes[~dupes["cn_code"].isin(["25231000", "25239000"])]
    if non_cement.empty:
        print("PASS  All duplicates are cement CN codes (25231000, 25239000).")
        print("      Grey and white clinker routes per country are the expected cause.")
    else:
        print("FAIL  Unexpected duplicates found outside cement CN codes:")
        print(non_cement.to_string(index=False))

Duplicate (country, cn_code) pairs in cbam_defaults: 29
PASS  All duplicates are cement CN codes (25231000, 25239000).
      Grey and white clinker routes per country are the expected cause.


In [19]:
# ============================================================
# Analytical gap check 1: CBAM countries with no trade flow data.
# Countries present in cbam_defaults but absent from trade_flows
# cannot contribute to any import exposure calculation.
# Namibia is the only known case.
# ============================================================
query = """
    SELECT d.country, COUNT(DISTINCT d.cn_code) AS cn_codes_in_defaults
    FROM cbam_defaults d
    LEFT JOIN trade_flows t ON d.country = t.country
    WHERE t.country IS NULL
    GROUP BY d.country
    ORDER BY d.country
"""
gaps = pd.read_sql(query, conn)
print(f"CBAM countries with no trade flow data: {len(gaps)}")
if not gaps.empty:
    print(gaps.to_string(index=False))

CBAM countries with no trade flow data: 1
country  cn_codes_in_defaults
Namibia                     3


In [20]:
# ============================================================
# Analytical gap check 2: CBAM countries with no grid CO2
# intensity data. Countries absent from grid_co2_intensity
# cannot have indirect emissions calculated.
# Curacao is the only known case.
# ============================================================
query = """
    SELECT DISTINCT d.country
    FROM cbam_defaults d
    LEFT JOIN grid_co2_intensity g ON d.country = g.country
    WHERE g.country IS NULL
    ORDER BY d.country
"""
no_grid = pd.read_sql(query, conn)
print(f"CBAM countries with no grid CO2 intensity data: {len(no_grid)}")
if not no_grid.empty:
    print(no_grid.to_string(index=False))

CBAM countries with no grid CO2 intensity data: 1
country
Curacao


In [21]:
# ============================================================
# Analytical gap check 3: Steel production route coverage.
#
# The three routes in steel_route_intensities (BF-BOF, DRI-EAF,
# Scrap-EAF) map to production_route_code values in cbam_defaults
# via a controlled code mapping:
#
#   BF-BOF    -> codes (C), (F), (C)/(F)
#   DRI-EAF   -> codes (D), (G)
#   Scrap-EAF -> codes (E), (H), (J), (E)/(H)
#
# DRI-EAF is expected to return zero matches because the EU
# Commission did not assign route codes (D) or (G) to any
# country in the defaults dataset. It is retained as a
# benchmark reference row only.
# ============================================================
query = """
    SELECT
        s.production_route,
        COUNT(DISTINCT d.country)  AS cbam_countries,
        COUNT(DISTINCT d.cn_code)  AS cn_codes
    FROM steel_route_intensities s
    LEFT JOIN cbam_defaults d
        ON (
               (s.production_route = 'BF-BOF'    AND d.production_route_code IN ('(C)', '(F)', '(C)/(F)'))
            OR (s.production_route = 'DRI-EAF'   AND d.production_route_code IN ('(D)', '(G)'))
            OR (s.production_route = 'Scrap-EAF' AND d.production_route_code IN ('(E)', '(H)', '(J)', '(E)/(H)'))
        )
    WHERE s.production_route != 'Global avg'
    GROUP BY s.production_route
    ORDER BY s.production_route
"""
route_coverage = pd.read_sql(query, conn)
print("Steel route intensity coverage in cbam_defaults:")
print(route_coverage.to_string(index=False))
print()
print("Note: DRI-EAF returning 0 matches is expected. The EU Commission")
print("did not assign route codes (D) or (G) to any country in the defaults.")

Steel route intensity coverage in cbam_defaults:
production_route  cbam_countries  cn_codes
          BF-BOF              26       148
         DRI-EAF               0         0
       Scrap-EAF               5       142

Note: DRI-EAF returning 0 matches is expected. The EU Commission
did not assign route codes (D) or (G) to any country in the defaults.


In [22]:
# ============================================================
# Join integrity check: global_exports -> country_crosswalk
# Every country in global_exports must resolve to a crosswalk
# row. The 20 countries absent from Comtrade results will
# simply have no rows here -- that is expected and documented.
# ============================================================
query = """
    SELECT g.country, COUNT(*) AS rows
    FROM global_exports g
    LEFT JOIN country_crosswalk c ON g.country = c.country
    WHERE c.country IS NULL
    GROUP BY g.country
    ORDER BY g.country
"""
unmatched = pd.read_sql(query, conn)

if unmatched.empty:
    n_ge = pd.read_sql(
        "SELECT COUNT(DISTINCT country) AS n FROM global_exports;", conn
    ).iloc[0]["n"]
    print(f"PASS  global_exports -> crosswalk: all {n_ge} countries matched")
else:
    print(f"FAIL  global_exports -> crosswalk: {len(unmatched)} unmatched country name(s):")
    print(unmatched.to_string(index=False))

# Report the 20 known-absent CBAM countries for the summary
query_absent = """
    SELECT c.country
    FROM country_crosswalk c
    WHERE c.name_cbam_defaults IS NOT NULL
      AND c.country NOT IN (SELECT DISTINCT country FROM global_exports)
    ORDER BY c.country
"""
absent = pd.read_sql(query_absent, conn)
print(f"\nCBAM countries absent from global_exports (no Comtrade data): {len(absent)}")
print(absent.to_string(index=False))

PASS  global_exports -> crosswalk: all 106 countries matched

CBAM countries absent from global_exports (no Comtrade data): 13
          country
       Bangladesh
Equatorial Guinea
          Eritrea
            Haiti
             Iraq
            Libya
      North Korea
     Sierra Leone
            Sudan
            Syria
           Taiwan
     Turkmenistan
        Venezuela


In [23]:
# ============================================================
# Join integrity check: steel_route_mix -> country_crosswalk
# All 119 CBAM countries should be present in steel_route_mix.
# A missing country would mean the route assignment in nb07
# Section 7 did not cover it.
# ============================================================
query = """
    SELECT s.country, COUNT(*) AS rows
    FROM steel_route_mix s
    LEFT JOIN country_crosswalk c ON s.country = c.country
    WHERE c.country IS NULL
    GROUP BY s.country
    ORDER BY s.country
"""
unmatched = pd.read_sql(query, conn)

if unmatched.empty:
    n_mix = pd.read_sql(
        "SELECT COUNT(DISTINCT country) AS n FROM steel_route_mix;", conn
    ).iloc[0]["n"]
    print(f"PASS  steel_route_mix -> crosswalk: all {n_mix} countries matched")
else:
    print(f"FAIL  steel_route_mix -> crosswalk: {len(unmatched)} unmatched country name(s):")
    print(unmatched.to_string(index=False))

# Confirm route_source breakdown matches what nb07 produced
query_sources = """
    SELECT route_source, COUNT(*) AS countries
    FROM steel_route_mix
    GROUP BY route_source
    ORDER BY route_source
"""
print(f"\nroute_source breakdown:\n{pd.read_sql(query_sources, conn).to_string(index=False)}")

PASS  steel_route_mix -> crosswalk: all 119 countries matched

route_source breakdown:
  route_source  countries
        direct         22
regional_proxy         97


In [24]:
# ============================================================
# Close the database connection cleanly.
# ============================================================
conn.close()
print(f"Connection closed. Database written to: {db_path.resolve()}")

Connection closed. Database written to: /Users/milcahmaryjoseph/Documents/GitHub/cbam-analysis/db/cbam.db


---
## Section 6: Observations

**Schema**
All 10 tables were created with explicit `PRIMARY KEY` and `FOREIGN KEY`
declarations before data was loaded. Foreign keys on `country` link all
country-level tables to `country_crosswalk`, producing a visible
relational diagram in DBeaver.

**Row counts**
All 8 tables loaded with row counts matching the clean CSVs exactly.

**Foreign key integrity**
`PRAGMA foreign_key_check` returned no violations across any table.
Every declared FK relationship is satisfied by the loaded data.

**Join integrity**
All CBAM default countries and all trade flow partners resolve cleanly
to the crosswalk. The trade flow to CBAM defaults match rate is
approximately 31%, which is expected given that EU member state partners
and non-CBAM CN codes make up the majority of trade flow rows.
The join produces extra rows for cement CN codes 25231000 and 25239000,
where grey and white clinker are assigned separate production route rows
per country in cbam_defaults. This is correct source behavior.

**Analytical gaps**
- **Namibia**: present in `cbam_defaults` (3 CN codes) but has no EU
  import trade flow data. Cannot contribute to exposure calculations.
- **Curacao**: present in `cbam_defaults` and `country_crosswalk` but
  absent from Ember grid data. Indirect emissions cannot be calculated.
- **DRI-EAF**: the EU Commission did not assign production route codes
  (D) or (G) to any country. The Worldsteel DRI-EAF intensity row is
  retained as a reference benchmark only and joins to zero default rows.
- 106 of 119 CBAM countries have Comtrade export data loaded. Coverage
  improved from 99 by pulling 2020-2024 and using the latest available
  year per (country, hs6_code) pair. The remaining 13 countries have no
  Comtrade data across any year in that range and are documented as hard
  gaps.